### Recurrent Neural Network (LSTM, V2)

This is a basic convolutional approach to sequence prediction using convolutions through TensorFlow!

We begin by important any relevant packages, modules, and the DataProcessor class (for our pre-processed data). Then we get our dataset ready.

In [2]:
import os, sys
sys.path.append(os.path.abspath('..'))  # add parent directory to sys.path
from data_cleanup import DataProcessor
from tensorflow.keras.layers import Dense, LSTM, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

# Define the windowing parameters
# Use 24 hours of history
INPUT_WINDOW = 72
# Predict the next 24 hours
OUTPUT_WINDOW = 24 

# Initialize the class
processor = DataProcessor(input_steps=INPUT_WINDOW, output_steps=OUTPUT_WINDOW)

# Run the pipeline
(X_train, y_train), (X_val, y_val), (X_test, y_test) = processor.load_and_process_data()

# Check the final shapes
print("\n--- Final Data Shapes ---")
print(f"X_train shape: {X_train.shape}  | y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}     | y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}    | y_test shape:  {y_test.shape}")

Step 1/5: Fetching, cleaning, and engineering features...


/opt/anaconda3/envs/ds_env/lib/python3.11/site-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)
/Users/macychen/GithubProjects/ECS171G13/data_cleanup.py:135: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='ffill')
/Users/macychen/GithubProjects/ECS171G13/data_cleanup.py:175: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df.resample('H').agg(agg_dict)


Step 2/5: Resampling data to hourly and setting 'Global_active_power' as target...
Step 3/5: Splitting data and applying scaler...
Step 4/5: Creating time-series windows...
Step 5/5: Data processing complete.

--- Final Data Shapes ---
X_train shape: (25832, 72, 8)  | y_train shape: (25832, 24)
X_val shape:   (3529, 72, 8)     | y_val shape:   (3529, 24)
X_test shape:  (4943, 72, 8)    | y_test shape:  (4943, 24)


/Users/macychen/GithubProjects/ECS171G13/data_cleanup.py:176: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_hourly = df_hourly.fillna(method='ffill')


Next, we want to build the model itself. Keras makes this very simple at a high-level, so tuning is also easy to do.

In [3]:
from tensorflow.keras.layers import Input, Dense, LSTM, Dropout, RepeatVector, TimeDistributed
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
model = Sequential()

model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))
model.add(LSTM(128, return_sequences=False)) 
model.add(Dropout(0.1))

model.add(RepeatVector(OUTPUT_WINDOW))

model.add(LSTM(128, return_sequences=True))
model.add(Dropout(0.1))

model.add(TimeDistributed(Dense(1)))

model.compile(optimizer=Adam(learning_rate=0.0005), 
              loss='mse', 
              metrics=['mae'])

model.summary()

2025-11-24 19:20:56.904977: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2025-11-24 19:20:56.905157: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2025-11-24 19:20:56.905168: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
I0000 00:00:1764040856.905484 31337909 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1764040856.905779 31337909 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │        70,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 24, 128)        │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 24, 1)          │           129 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 201,857 (788.50 KB)

 Trainable params: 201,857 (788.50 KB)

 Non-trainable params: 0 (0.00 B)

Now we want to fit the model, train it, and determine an error metric.

In [4]:
es = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50, 
    batch_size=32,
    callbacks=[es],
    verbose=1
)

sample_idx = 50
input_seq = X_test[sample_idx]  # Shape (72, features)
actual_output = y_test[sample_idx] # Shape (24, 1)

pred_scaled = model.predict(input_seq.reshape(1, INPUT_WINDOW, X_test.shape[2]))

pred_unscaled = processor.inverse_transform_predictions(pred_scaled[0].flatten())
actual_unscaled = processor.inverse_transform_predictions(actual_output.flatten())

mse = mean_squared_error(actual_unscaled, pred_unscaled)
rmse = np.sqrt(mse)

print(f'\n--- Model Evaluation ---')
print(f'Test Set RMSE: {rmse:.4f} kW')

Epoch 1/50


2025-11-24 19:21:03.222280: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


808/808 ━━━━━━━━━━━━━━━━━━━━ 28s 28ms/step - loss: 0.0179 - mae: 0.1036 - val_loss: 0.0141 - val_mae: 0.0905
Epoch 2/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 20s 24ms/step - loss: 0.0142 - mae: 0.0883 - val_loss: 0.0132 - val_mae: 0.0872
Epoch 3/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - loss: 0.0135 - mae: 0.0853 - val_loss: 0.0130 - val_mae: 0.0852
Epoch 4/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 20s 25ms/step - loss: 0.0130 - mae: 0.0834 - val_loss: 0.0126 - val_mae: 0.0843
Epoch 5/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 20s 24ms/step - loss: 0.0126 - mae: 0.0819 - val_loss: 0.0129 - val_mae: 0.0867
Epoch 6/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 20s 24ms/step - loss: 0.0122 - mae: 0.0803 - val_loss: 0.0131 - val_mae: 0.0857
Epoch 7/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - loss: 0.0119 - mae: 0.0790 - val_loss: 0.0127 - val_mae: 0.0828
Epoch 8/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - loss: 0.0115 - mae: 0.0775 - val_loss: 0.0128 - val_mae: 0.0837
Epoch 9/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 20s 24ms/st

In [ ]:
# Graph 1: Loss Curves (MSE)

plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss (MSE)', color='blue')
plt.plot(history.history['val_loss'], label='Validation Loss (MSE)', color='orange')
plt.title('Training and Validation Loss over Epochs (repeat)')
plt.xlabel('Epochs')
plt.ylabel('Loss (Mean Squared Error)')
plt.legend()
plt.grid(True)
plt.show()

# Graph 2: MAE Curves

plt.figure(figsize=(12, 6))
plt.plot(history.history['mae'], label='Training MAE', color='green')
plt.plot(history.history['val_mae'], label='Validation MAE', color='red')
plt.title('Training and Validation Mean Absolute Error (repeat)')
plt.xlabel('Epochs')
plt.ylabel('MAE (Scaled)')
plt.legend()
plt.grid(True)
plt.show()

# Graph 3: Actual vs Predicted (Time Series)

# We take the first 200 hours for a clean zoom-in (like the uploaded image)
subset_n = 200 

# Extract just the 1st hour prediction from each window (index 0)
# This reconstructs a continuous timeline
actual_trace = unscaled_y_test[:subset_n, 0]
pred_trace = unscaled_predictions[:subset_n, 0]

plt.figure(figsize=(14, 6))
plt.plot(actual_trace, label='Actual Power (kW)', color='black', linewidth=2)
plt.plot(pred_trace, label='LSTM Predicted (kW)', color='cyan', linestyle='--')
plt.title(f'Actual vs Predicted Global Active Power (First {subset_n} Test Hours) (repeat)')
plt.xlabel('Time (Hours)')
plt.ylabel('Global Active Power (kW)')
plt.legend()
plt.grid(True)
plt.show()

# Graph 4: 24-Hour Rolling Comparison

import matplotlib.pyplot as plt

sample_idx = 50

# Get the actual 24-hour sequence for this sample
real_24h = unscaled_y_test[sample_idx]

# Get the predicted 24-hour sequence for this sample
pred_24h = unscaled_predictions[sample_idx]

plt.figure(figsize=(10, 5))
plt.plot(real_24h, label='Actual (Real)', marker='o', color='black')
plt.plot(pred_24h, label='Predicted (Repeat LSTM)', marker='x', linestyle='--', color='cyan')

plt.title(f'24-Hour Energy Forecast (Sample #{sample_idx})')
plt.xlabel('Hour of the Day')
plt.ylabel('Global Active Power (kW)')
plt.legend()
plt.grid(True)
plt.show()

In [6]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import numpy as np

FOLDS = 3
tscv = TimeSeriesSplit(n_splits=FOLDS)

cv_mse = []
cv_rmse = []
cv_mae = []

print("\nStarting Cross-Validation")

fold = 1
for train_idx, val_idx in tscv.split(X_train):
    print(f'\n--- Fold {fold} ---')
    
    X_tr, X_val_fold = X_train[train_idx], X_train[val_idx]
    y_tr, y_val_fold = y_train[train_idx], y_train[val_idx]
    
    # Define model
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))
    model.add(LSTM(128, return_sequences=False))
    model.add(Dropout(0.1))
    model.add(RepeatVector(OUTPUT_WINDOW))
    model.add(LSTM(128, return_sequences=True))
    model.add(Dropout(0.1))
    model.add(TimeDistributed(Dense(1)))
    
    model.compile(
        optimizer=Adam(learning_rate=0.0005),
        loss='mse',
        metrics=['mae']
    )
    
    es = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
    
    # Fit model
    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val_fold, y_val_fold),
        epochs=50,
        batch_size=32,
        callbacks=[es],
        verbose=1
    )
    
    # Evaluate on validation fold
    y_pred = model.predict(X_val_fold)
    
    y_pred_unscaled = processor.inverse_transform_predictions(y_pred.flatten())
    y_val_unscaled = processor.inverse_transform_predictions(y_val_fold.flatten())
    
    # Calculate metrics
    fold_mse = mean_squared_error(y_val_unscaled, y_pred_unscaled)
    fold_rmse = np.sqrt(mse)
    fold_mae = np.mean(np.abs(y_val_unscaled - y_pred_unscaled))
    
    print(f"Fold {fold} MSE: {fold_mse:.4f}, RMSE: {fold_rmse:.4f}, MAE: {fold_mae:.4f}")
    
    cv_rmse.append(fold_rmse)
    cv_mse.append(fold_mse)
    cv_mae.append(fold_mae)

    fold += 1

# Cross-Validation Summary
print("\nCross-Validation Summary")
print(f"Average MSE: {np.mean(cv_mse):.4f}")
print(f"Average RMSE: {np.mean(cv_rmse):.4f}")


Starting Cross-Validation

--- Fold 1 ---
Epoch 1/50
202/202 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - loss: 0.0217 - mae: 0.1156 - val_loss: 0.0237 - val_mae: 0.1187
Epoch 2/50
202/202 ━━━━━━━━━━━━━━━━━━━━ 7s 32ms/step - loss: 0.0207 - mae: 0.1126 - val_loss: 0.0228 - val_mae: 0.1199
Epoch 3/50
202/202 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - loss: 0.0203 - mae: 0.1117 - val_loss: 0.0228 - val_mae: 0.1204
Epoch 4/50
202/202 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - loss: 0.0198 - mae: 0.1099 - val_loss: 0.0222 - val_mae: 0.1218
Epoch 5/50
202/202 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step - loss: 0.0187 - mae: 0.1065 - val_loss: 0.0193 - val_mae: 0.1064
Epoch 6/50
202/202 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - loss: 0.0165 - mae: 0.0981 - val_loss: 0.0181 - val_mae: 0.1021
Epoch 7/50
202/202 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - loss: 0.0154 - mae: 0.0936 - val_loss: 0.0187 - val_mae: 0.1027
Epoch 8/50
202/202 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - loss: 0.0147 - mae: 0.0911 - val_loss: 0.0184 - val_mae: 0.1011
Epoc